**TAHAP 1**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
folder_path = '/content/drive/MyDrive/cookiedong-portfolio'
os.makedirs(folder_path, exist_ok=True)

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

from google.colab import files
uploaded = files.upload()  # upload file data terbaru (dengan Produk_ID & HPP)

filename = list(uploaded.keys())[0]

if filename.endswith('.csv'):
    df = pd.read_csv(filename, sep=';')
elif filename.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(filename)

print(f"Jumlah baris: {df.shape[0]}, Jumlah kolom: {df.shape[1]}")
df.head(10)

IndexError: list index out of range

In [ ]:
df.info()

In [ ]:
for col in ['Customer', 'Channel', 'Menu']:
    print(f"\n=== {col} ===")
    print(f"Jumlah unik: {df[col].nunique()}")
    print(df[col].unique())

In [ ]:
def parse_rupiah(nilai):
    if pd.isna(nilai):
        return np.nan
    nilai = str(nilai)
    nilai = nilai.replace('Rp', '').strip()
    nilai = nilai.replace('.', '')      # hapus pemisah ribuan
    nilai = nilai.replace(',', '.')     # ganti koma desimal jadi titik
    return float(nilai)

df['HPP'] = df['HPP'].apply(parse_rupiah)

print(df[['Menu', 'HPP']].head(10))
print(f"\nTipe data HPP setelah parsing: {df['HPP'].dtype}")

In [ ]:
cek_hpp = df.groupby('Produk_ID')['HPP'].nunique()
produk_hpp_tidak_konsisten = cek_hpp[cek_hpp > 1]

print(f"Jumlah Produk_ID dengan HPP tidak konsisten: {len(produk_hpp_tidak_konsisten)}")
if len(produk_hpp_tidak_konsisten) > 0:
    print(df[df['Produk_ID'].isin(produk_hpp_tidak_konsisten.index)][['Produk_ID', 'Menu', 'HPP']].drop_duplicates())

In [ ]:
menu_mapping = {
    'Fudgy Brownies Filling Lotus Biscoff 10 x 10': 'Fudgy Brownies Lotus Biscoff Filling 10 x 10',
    'Fudgy Brochizu (20x10)': 'Fudgy Brownies Brochizu 20 x 10',
    'Banana Bread Choco Almond': 'Banana Bread Choco Almond 10 x 10',
}
df['Menu'] = df['Menu'].replace(menu_mapping)
print(f"Total menu unik setelah standarisasi: {df['Menu'].nunique()}")

In [ ]:
cek_konsistensi = df.groupby('Produk_ID')['Menu'].nunique()
tidak_konsisten = cek_konsistensi[cek_konsistensi > 1]

print(f"Jumlah Produk_ID dengan >1 nama Menu: {len(tidak_konsisten)}")
if len(tidak_konsisten) > 0:
    print(df[df['Produk_ID'].isin(tidak_konsisten.index)][['Produk_ID', 'Menu']].drop_duplicates())

In [ ]:
df['cek_subtotal'] = df['Qty'] * df['Harga']
df['selisih'] = df['cek_subtotal'] - df['Subtotal']

baris_tidak_cocok = df[df['selisih'] != 0]
print(f"Baris tidak cocok: {len(baris_tidak_cocok)}")

In [ ]:
print(f"Jumlah duplikat sebelum: {df.duplicated().sum()}")
df = df.drop_duplicates(keep='first').reset_index(drop=True)
print(f"Jumlah baris setelah hapus duplikat: {len(df)}")

In [ ]:
# --- Checkpoint: simpan data level item (sudah termasuk Laba & Margin) ---
df.to_csv(f'{folder_path}/order_items_clean.csv', index=False)
print("Checkpoint tersimpan: order_items_clean.csv")
print(f"Jumlah baris: {len(df)}, Jumlah kolom: {df.shape[1]}")

**TAHAP 2**

In [ ]:
df = df.drop(columns=['cek_subtotal', 'selisih'])

In [ ]:
df['Tanggal Order'] = pd.to_datetime(df['Tanggal Order'], format='%d/%m/%Y')
print(df['Tanggal Order'].head())
print(f"Tipe data: {df['Tanggal Order'].dtype}")

In [ ]:
df['Minggu'] = df['Tanggal Order'].dt.isocalendar().week
df['Bulan'] = df['Tanggal Order'].dt.month_name()
df['Hari'] = df['Tanggal Order'].dt.day_name()

In [ ]:
def kategorikan_menu(nama_menu):
    nama = nama_menu.lower()
    if 'scoopable' in nama:
        return 'Scoopable'
    elif 'fudgy' in nama or 'brownies' in nama:
        return 'Fudgy Brownies'
    elif 'banana bread' in nama:
        return 'Banana Bread'
    elif 'cookies' in nama:
        return 'Cookies'
    else:
        return 'Lainnya'

df['Kategori Menu'] = df['Menu'].apply(kategorikan_menu)
print(df['Kategori Menu'].value_counts())

In [ ]:
# Total HPP untuk baris ini = HPP per unit x Qty
df['Total_HPP'] = df['HPP'] * df['Qty']

# Laba = Subtotal (omzet) - Total HPP
df['Laba'] = df['Subtotal'] - df['Total_HPP']

# Margin % = Laba / Subtotal x 100
df['Margin_Persen'] = (df['Laba'] / df['Subtotal'] * 100).round(2)

df[['Menu', 'Qty', 'Harga', 'Subtotal', 'HPP', 'Total_HPP', 'Laba', 'Margin_Persen']].head(10)

In [ ]:
total_omzet = df['Subtotal'].sum()
total_hpp = df['Total_HPP'].sum()
total_laba = df['Laba'].sum()
margin_keseluruhan = (total_laba / total_omzet * 100)

print(f"Total Omzet: Rp {total_omzet:,.0f}")
print(f"Total HPP: Rp {total_hpp:,.0f}")
print(f"Total Laba: Rp {total_laba:,.0f}")
print(f"Margin keseluruhan: {margin_keseluruhan:.1f}%")

In [ ]:
margin_negatif = df[df['Laba'] < 0]
print(f"Jumlah baris dengan laba negatif: {len(margin_negatif)}")
if len(margin_negatif) > 0:
    print(margin_negatif[['Menu', 'Harga', 'HPP', 'Laba']].drop_duplicates())

In [ ]:
df.to_csv(f'{folder_path}/order_items_clean.csv', index=False)
print("Checkpoint tersimpan: order_items_clean.csv")
print(f"Jumlah baris: {len(df)}, Jumlah kolom: {df.shape[1]}")

**TAHAP 3**

In [ ]:
orders = df.groupby(['Customer', 'Channel', 'Tanggal Order'], as_index=False).agg(
    Total_Qty=('Qty', 'sum'),
    Total_Omzet=('Subtotal', 'sum'),
    Total_HPP=('Total_HPP', 'sum'),
    Total_Laba=('Laba', 'sum'),
    Jumlah_Item=('Menu', 'count')
)

orders['Margin_Persen'] = (orders['Total_Laba'] / orders['Total_Omzet'] * 100).round(2)

orders.insert(0, 'Order_ID', ['ORD-' + str(i+1).zfill(3) for i in range(len(orders))])

orders['Minggu'] = orders['Tanggal Order'].dt.isocalendar().week
orders['Bulan'] = orders['Tanggal Order'].dt.month_name()
orders['Hari'] = orders['Tanggal Order'].dt.day_name()

print(f"Jumlah order: {len(orders)}")
orders.head(10)

In [ ]:
# --- Merge Order_ID kembali ke tabel item ---
df = df.merge(
    orders[['Order_ID', 'Customer', 'Channel', 'Tanggal Order']],
    on=['Customer', 'Channel', 'Tanggal Order'],
    how='left'
)

print(f"Baris tanpa Order_ID: {df['Order_ID'].isnull().sum()}")

In [ ]:
# --- Export final kedua tabel ---
df.to_csv(f'{folder_path}/order_items_clean.csv', index=False)
orders.to_csv(f'{folder_path}/orders_clean.csv', index=False)

print("Kedua file berhasil diperbarui dengan data Laba & Margin")

In [ ]:
# Install library untuk akses Google Sheets
!pip install gspread gspread-dataframe --quiet

from google.colab import auth
auth.authenticate_user()

import gspread
from gspread_dataframe import set_with_dataframe
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

def upload_to_sheets(df, nama_sheet):
    sh = gc.create(nama_sheet)
    worksheet = sh.get_worksheet(0)
    set_with_dataframe(worksheet, df)
    print(f"{nama_sheet} berhasil dibuat: {sh.url}")

upload_to_sheets(orders, 'CookieDong - Orders')
upload_to_sheets(df, 'CookieDong - Order Items')
upload_to_sheets(rfm, 'CookieDong - RFM Customer')